# 📖 Notebook 4: Document Versioning and History

Every collaborative editor needs version history — the ability to see what changed, when, and by whom, and to restore previous versions. In this notebook, we explore how **snapshots** (compaction) and **operation logs** work together to provide efficient versioning.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why raw operation logs grow too large over time
- How snapshots (compaction) reduce storage and speed up loading
- How to implement version history and restore
- The trade-offs between storage, performance, and history granularity

## 🛠️ Setup

```bash
cd system-designs/google-docs
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import json
import time
import websockets
import asyncio

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "googledocs_demo",
    "user": "demo",
    "password": "demo"
}

WS_URL = "ws://localhost:8765"

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def db_query(sql, params=None):
    conn = get_db()
    try:
        with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
            cur.execute(sql, params)
            return cur.fetchall()
    finally:
        conn.close()

try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

## 🤔 The Problem: Operations Grow Forever

Every keystroke creates an operation. For an active document:

```
1 user typing at 5 keys/second × 3600 seconds = 18,000 operations per hour
10 users × 8 hours = 1,440,000 operations per day
```

When a new user opens the document, the server would need to:
1. Load **all** operations from the database
2. Replay them one by one to reconstruct the current text

For 1.4 million operations, that's slow and expensive!

**The solution**: periodically take a **snapshot** — save the full document text and discard (or archive) the old operations. New users only need the latest snapshot + any operations since then.

In [ ]:
# Let's visualize how operation counts grow vs snapshots

hours = list(range(1, 25))
ops_per_hour = 18000  # 10 users × 5 keystrokes/sec × 3600 sec / 10

# Without snapshots: operations accumulate
no_snapshot_ops = [ops_per_hour * h for h in hours]

# With snapshots every 2 hours: ops reset after each snapshot
with_snapshot_ops = [ops_per_hour * (h % 2 or 2) for h in hours]

print("📊 Operations to Replay When Loading Document")
print("=" * 60)
print(f"{'Hour':>4}  {'No Snapshots':>15}  {'Snapshot/2hr':>15}  Savings")
print("-" * 60)
for h in [1, 4, 8, 12, 24]:
    no_snap = no_snapshot_ops[h-1]
    with_snap = with_snapshot_ops[h-1]
    savings = ((no_snap - with_snap) / no_snap * 100) if no_snap > 0 else 0
    print(f"{h:>4}  {no_snap:>12,} ops  {with_snap:>12,} ops  {savings:.0f}%")

print()
print("💡 After 24 hours, snapshots reduce ops to replay by 94%!")
print("   This is why Google Docs periodically compacts operations.")

## 📸 What's a Snapshot?

A **snapshot** is a frozen copy of the complete document text at a specific point in time.

```
Operations:  [op1, op2, op3, ..., op50]  →  Snapshot v1: "Full document text..."
Operations:  [op51, op52, ..., op100]     →  Snapshot v2: "Updated full text..."
```

To load the document, you only need:
- The **latest snapshot** (full text)
- Plus any **operations after that snapshot** (usually very few)

Let's look at the snapshots in our database.

In [ ]:
# View existing snapshots
snapshots = db_query("""
    SELECT s.document_id, d.title, s.version, 
           LENGTH(s.content) as content_length,
           s.op_count, u.display_name as created_by,
           s.created_at
    FROM snapshots s
    JOIN documents d ON s.document_id = d.id
    JOIN users u ON s.created_by = u.id
    ORDER BY s.document_id, s.version
""")

print("📸 Existing Snapshots")
print("=" * 80)
print(f"{'Doc':>3}  {'Title':<35} {'Ver':>3}  {'Chars':>6}  {'Ops':>4}  {'By':<15}")
print("-" * 80)
for s in snapshots:
    title = s['title'][:33] + '..' if len(s['title']) > 33 else s['title']
    print(f"{s['document_id']:>3}  {title:<35} {s['version']:>3}  {s['content_length']:>6}  {s['op_count']:>4}  {s['created_by']:<15}")

print(f"\n💡 Version 0 = empty document (initial state)")
print(f"   Version 1 = first saved state (from seed data)")

In [ ]:
# View the content of snapshots for document 1
doc1_snapshots = db_query("""
    SELECT version, content, op_count, created_at
    FROM snapshots
    WHERE document_id = 1
    ORDER BY version
""")

print("📄 Document 1: 'Meeting Notes — Q1 Planning'")
print("=" * 60)
for snap in doc1_snapshots:
    print(f"\n📸 Version {snap['version']} ({snap['op_count']} ops compacted):")
    print(f"   Created: {snap['created_at']}")
    if snap['content']:
        for line in snap['content'].split('\n'):
            print(f"   │ {line}")
    else:
        print(f"   │ (empty document)")

## 🔄 Loading a Document: Snapshot + Replay

When a user opens a document, the server:
1. Loads the **latest snapshot** from the `snapshots` table
2. Loads any **operations after that snapshot** from the `operations` table
3. Replays those operations on top of the snapshot

This is much faster than replaying ALL operations from the beginning.

In [ ]:
def load_document_from_db(doc_id):
    """Load a document using snapshot + replay.
    
    This is exactly what the doc server does when loading a document.
    """
    # Step 1: Get the latest snapshot
    snapshots = db_query(
        "SELECT version, content FROM snapshots "
        "WHERE document_id = %s ORDER BY version DESC LIMIT 1",
        (doc_id,)
    )
    
    if snapshots:
        text = snapshots[0]["content"]
        version = snapshots[0]["version"]
        print(f"  📸 Loaded snapshot v{version} ({len(text)} chars)")
    else:
        text = ""
        version = 0
        print(f"  📸 No snapshot found, starting empty")
    
    # Step 2: Get operations since the snapshot
    ops = db_query(
        "SELECT op_type, position, content, length FROM operations "
        "WHERE document_id = %s AND version >= %s ORDER BY id",
        (doc_id, version)
    )
    
    print(f"  🔄 Found {len(ops)} operations to replay")
    
    # Step 3: Replay operations
    for op in ops:
        pos = op["position"]
        if op["op_type"] == "insert":
            text = text[:pos] + (op["content"] or "") + text[pos:]
        elif op["op_type"] == "delete":
            length = op["length"] or 1
            text = text[:pos] + text[pos + length:]
    
    print(f"  ✅ Document loaded: {len(text)} chars")
    return text, version

print("Loading Document 1:")
text, version = load_document_from_db(1)
print(f"\n📄 Content preview:")
print(text[:200])

## 💾 Creating Snapshots via the Server

Let's connect to the doc server, make some edits, and create a snapshot.

In [ ]:
async def create_edits_and_snapshot(doc_id, user_id=1):
    """Connect, make edits, and save a snapshot."""
    async with websockets.connect(WS_URL) as ws:
        # Connect
        await ws.send(json.dumps({"type": "connect", "user_id": user_id}))
        await ws.recv()  # connected
        
        # Join document
        await ws.send(json.dumps({"type": "join_doc", "document_id": doc_id}))
        doc_state = json.loads(await ws.recv())  # doc_state
        await ws.recv()  # presence_list
        
        doc_text = doc_state["text"]
        print(f"📄 Joined doc {doc_id} (version {doc_state['version']})")
        print(f"   Current length: {len(doc_text)} chars")
        
        # Make some edits
        edits = [
            "\n\n--- New Section ---",
            "\nThis text was added to demonstrate versioning.",
            "\nTimestamp: " + time.strftime("%Y-%m-%d %H:%M:%S"),
        ]
        
        for edit_text in edits:
            pos = len(doc_text)
            await ws.send(json.dumps({
                "type": "edit",
                "document_id": doc_id,
                "op_type": "insert",
                "position": pos,
                "content": edit_text,
            }))
            await ws.recv()  # ack
            doc_text += edit_text
            print(f"   ✏️  Inserted {len(edit_text)} chars")
        
        # Save a snapshot
        await ws.send(json.dumps({
            "type": "save_snapshot",
            "document_id": doc_id,
        }))
        resp = json.loads(await ws.recv())
        print(f"   📸 Snapshot saved: version {resp.get('version', '?')}")
        
        return resp

result = await create_edits_and_snapshot(2)  # Use document 2

In [ ]:
# Check what snapshots exist now
doc2_snapshots = db_query("""
    SELECT version, LENGTH(content) as chars, op_count, created_at
    FROM snapshots
    WHERE document_id = 2
    ORDER BY version
""")

print("📸 Snapshots for Document 2:")
print(f"{'Version':>7}  {'Chars':>6}  {'Ops Compacted':>14}  Created")
print("-" * 60)
for s in doc2_snapshots:
    print(f"{s['version']:>7}  {s['chars']:>6}  {s['op_count']:>14}  {s['created_at']}")

## 🕐 Version History and Restore

With snapshots, we can implement **version history** — showing users what the document looked like at each saved point, and letting them restore to a previous version.

In [ ]:
async def get_version_history(doc_id, user_id=1):
    """Fetch version history from the server."""
    async with websockets.connect(WS_URL) as ws:
        await ws.send(json.dumps({"type": "connect", "user_id": user_id}))
        await ws.recv()
        
        await ws.send(json.dumps({"type": "join_doc", "document_id": doc_id}))
        await ws.recv()  # doc_state
        await ws.recv()  # presence_list
        
        await ws.send(json.dumps({
            "type": "get_history",
            "document_id": doc_id,
        }))
        resp = json.loads(await ws.recv())
        return resp

history = await get_version_history(2)

print("🕐 Version History for Document 2:")
print("=" * 60)
for v in history.get("versions", []):
    content_preview = v["content"][:80] + "..." if len(v["content"]) > 80 else v["content"]
    print(f"\n  📸 Version {v['version']}:")
    print(f"     Created by: {v['created_by']}")
    print(f"     Created at: {v['created_at']}")
    print(f"     Ops compacted: {v['op_count']}")
    print(f"     Content: {content_preview}")

In [ ]:
# Demonstrate version restore
async def restore_version(doc_id, version, user_id=1):
    """Restore a document to a previous version."""
    async with websockets.connect(WS_URL) as ws:
        await ws.send(json.dumps({"type": "connect", "user_id": user_id}))
        await ws.recv()
        
        await ws.send(json.dumps({"type": "join_doc", "document_id": doc_id}))
        state = json.loads(await ws.recv())  # doc_state
        await ws.recv()  # presence_list
        
        print(f"Current document ({len(state['text'])} chars):")
        print(f"  '{state['text'][:100]}...'")
        print()
        
        await ws.send(json.dumps({
            "type": "restore_version",
            "document_id": doc_id,
            "version": version,
        }))
        
        # Read the response(s)
        resp = json.loads(await ws.recv())
        # May receive doc_state broadcast first
        if resp["type"] == "doc_state":
            print(f"Restored to version {version} ({len(resp['text'])} chars):")
            print(f"  '{resp['text'][:100]}...' " if len(resp['text']) > 100 else f"  '{resp['text']}'")
            resp = json.loads(await ws.recv())
        
        if resp["type"] == "version_restored":
            print(f"\n✅ Restored version {resp['restored_version']} → new version {resp['new_version']}")

print("Restoring Document 2 to version 1 (original content):")
print("=" * 60)
await restore_version(2, 1)

## 🗜️ Compaction Strategy

In a production system like Google Docs, compaction (snapshotting) happens strategically:

| Trigger | When | Why |
|---------|------|-----|
| **Operation count** | Every N ops (e.g., 50) | Prevents unbounded growth |
| **Idle document** | When last editor disconnects | Safe — no concurrent ops |
| **Manual save** | User clicks "Save" | User-initiated checkpoint |
| **Scheduled** | Periodic background job | Catches long-running sessions |

Our server auto-snapshots every **50 operations** and when the **last user disconnects**.

In [ ]:
# Let's look at the storage savings from compaction

# Count operations vs snapshot sizes
op_stats = db_query("""
    SELECT document_id, COUNT(*) as op_count,
           SUM(LENGTH(COALESCE(content, ''))) as total_content_bytes
    FROM operations
    GROUP BY document_id
""")

snap_stats = db_query("""
    SELECT document_id, COUNT(*) as snap_count,
           SUM(LENGTH(content)) as total_snap_bytes,
           MAX(version) as latest_version
    FROM snapshots
    GROUP BY document_id
""")

print("📊 Storage: Operations vs Snapshots")
print("=" * 60)
print(f"{'Doc':>3}  {'Ops':>5}  {'Op Bytes':>10}  {'Snaps':>5}  {'Snap Bytes':>10}")
print("-" * 60)

for op in op_stats:
    doc_id = op['document_id']
    snap = next((s for s in snap_stats if s['document_id'] == doc_id), None)
    snap_count = snap['snap_count'] if snap else 0
    snap_bytes = snap['total_snap_bytes'] if snap else 0
    print(f"{doc_id:>3}  {op['op_count']:>5}  {op['total_content_bytes'] or 0:>10}  {snap_count:>5}  {snap_bytes or 0:>10}")

print()
print("💡 Snapshots are more space-efficient because they don't store")
print("   the per-operation metadata (type, position, user, timestamp).")
print("   After compaction, old ops can be archived or deleted.")

## 🏗️ Production Considerations

### Google Docs' Approach

```
Document Service
     │
     ├─── On every edit: append to operations log (fast)
     │
     ├─── Every 50 ops: auto-snapshot (background)
     │
     ├─── On last disconnect: final snapshot + compaction
     │
     └─── Version metadata: stored in Document Metadata DB
          (which snapshot version is "current")
```

### Key Design Decisions

| Decision | Trade-off |
|----------|----------|
| Snapshot frequency | More often = faster loads, but more storage + CPU |
| Keep old operations | Yes = full audit trail, but storage grows |
| Version retention | Keep all = unlimited undo, but more storage |
| Compaction timing | During idle = safe; during editing = risky |

### Billions of Documents

With billions of documents at ~50KB each:
- Raw storage: **50 TB** just for current document text
- With operations + versions: could be **10-100× more**
- Compaction is **essential** to keep storage manageable

## 🧹 Cleanup

In [ ]:
print("🧹 No cleanup needed — snapshots are part of the demo data.")
print("   To fully reset, run: docker-compose down -v && docker-compose up -d")

## 📚 Summary

### Key Takeaways

1. **Operations grow unbounded** — every keystroke is logged, leading to millions of ops per document
2. **Snapshots (compaction)** save the full document text periodically, enabling fast loading
3. **Loading = latest snapshot + replay** — only replay ops since the last snapshot
4. **Version history** = a chain of snapshots, each with the full document at that point
5. **Restore** = create a new snapshot with the content from an old version

### For System Design Interviews

- Always mention **compaction/snapshots** when discussing operation logs
- Explain the **trade-off**: snapshot frequency vs storage vs load time
- Note that compaction should happen when the document is **idle** (no active editors)
- Version history is built **on top of** snapshots — not a separate system

### What We've Covered in This Lab

| Notebook | Topic | Key Concept |
|----------|-------|-------------|
| 1 | Operational Transformation | Transform concurrent ops to preserve intent |
| 2 | CRDTs | Order-independent operations, no central server |
| 3 | Real-Time Collaboration | WebSockets, presence, scaling with consistent hashing |
| 4 | Versioning & History | Snapshots, compaction, version restore |

Together, these form the core of a Google Docs-style collaborative document editor. 🎉